## 21. اعتبارسنجی مختصات و نقشه

پیش از ساخت نقشه:

- Missing coordinates را گزارش کنید.
- Latitude و Longitude جابه‌جا نشده باشند.
- نقاط خارج از محدوده ایران شناسایی شوند.
- نقاط خارج از محدوده شهر بررسی شوند.
- تعداد مختصات تکراری غیرعادی بررسی شود.
- دقت مکانی و احتمال ناشناس‌سازی مختصات توضیح داده شود.
- از نمایش نقطه‌ای یک میلیون رکورد خودداری شود.

روش‌های مناسب‌تر:

- Aggregation محله‌ای
- Hexbin
- Grid aggregation
- Sample کنترل‌شده
- Choropleth در صورت وجود مرزهای معتبر

In [1]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point

In [3]:
df = pd.read_feather("../Outputs/19_df.feather")

In [9]:
world = gpd.read_file(
    "https://naturalearth.s3.amazonaws.com/10m_cultural/ne_10m_admin_0_countries.zip"
)

iran = world[world["ADMIN"] == "Iran"].geometry.iloc[0]

In [4]:
valid_coords = (
    df['location_latitude'].notna() &
    df['location_longitude'].notna()
)

In [14]:
geo_df = gpd.GeoDataFrame(
    df.copy(),
    geometry=gpd.points_from_xy(
        df['location_longitude'],
        df['location_latitude']
    ),
    crs='EPSG:4326'
)

In [15]:
geo_df = geo_df.to_crs('EPSG:3857')
iran = gpd.GeoSeries([iran], crs='EPSG:4326').to_crs('EPSG:3857').iloc[0]

geo_df = geo_df.to_crs('EPSG:3857')
iran = gpd.GeoSeries([iran], crs='EPSG:4326').to_crs('EPSG:3857').iloc[0]

In [16]:
geo_df['location_area'] = geo_df.geometry.buffer(
    geo_df['location_radius'].fillna(0)
)

In [17]:
geo_df['coordinates_outside_iran'] = ~geo_df['location_area'].intersects(iran)

In [19]:
df['coordinates_outside_iran'].sum()

np.int64(4061)

In [13]:
outside_iran = df[
    df['coordinates_outside_iran']
]

outside_iran[
    [
        'city_slug',
        'neighborhood_slug',
        'location_latitude',
        'location_longitude',
        'location_radius'
    ]
]

,city_slug,neighborhood_slug,location_latitude,location_longitude,location_radius
233,bandar-abbas,NaN,25.667522,56.704559,<NA>
512,bushehr,NaN,29.790997,49.888100,<NA>
819,bandar-kangan,NaN,27.752630,52.037251,<NA>
860,bandar-abbas,NaN,26.367264,56.068039,<NA>
1032,kish,NaN,26.173927,54.032135,<NA>
...,...,...,...,...,...
999098,bandar-abbas,NaN,27.156309,56.265793,<NA>
999283,kish,NaN,26.198574,53.876266,<NA>
999401,nowshahr,NaN,36.640568,51.830791,<NA>
999469,zanjan,NaN,37.719311,49.246731,<NA>
